# LightGBM ハイパーパラメータチューニング

- **目的**: 目先のスコア上げ。特徴量は変えず **LGB のパラメータだけ** を Optuna で探索する。
- **データ・CV**: `train_baseline.ipynb` と同じ（`lib.get_baseline_data()` 38 特徴・時系列 4-fold CV）。
- **探索**: `n_estimators`, `learning_rate`, `num_leaves`, `min_data_in_leaf`, `feature_fraction`, `reg_alpha`, `reg_lambda`, `max_depth` など。
- ベースライン CV AUC は約 **0.760**。ここではチューニングで数ポイント〜0.01 程度の改善を狙う。

**パソコンを閉じても動かしたい（裏で回す）**  
同じ処理を **`run_lgb_tuning.py`** で実行できる。ターミナルでプロジェクトルートに移動してから:

```bash
# 例: 80 trials をバックグラウンドで実行し、ログを lgb_tuning.log に保存
nohup python run_lgb_tuning.py 80 > lgb_tuning.log 2>&1 &
# 進捗を見る: tail -f lgb_tuning.log
```

- **tmux** や **screen** を使う場合は、その中で `python run_lgb_tuning.py 80` を実行すれば、ターミナルを閉じてもプロセスは残る（セッションを detach しておく）。
- ノートPC で**蓋を閉じるとスリープ**するため、その間は止まる。ずっと回し続けたい場合はサーバーやクラウドで実行するか、スリープを無効にする必要がある。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    raise ImportError("Optuna が必要です: pip install optuna")

OUTPUT_DIR = "outputs"
os.makedirs(os.path.join(OUTPUT_DIR, "submissions"), exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)
print("Setup complete.")

Setup complete.


In [2]:
# データ・特徴量・時系列CV（ベースラインと同じ）
train, test = get_baseline_data()
FEATURES = BASELINE_FEATURES

VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

X = train[FEATURES]
y = train["target"].values
X_test = test[FEATURES]

print(f"Train: {len(train):,}, Test: {len(test):,}, Features: {len(FEATURES)}")
print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")

Train: 653,507, Test: 40,716, Features: 38
時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])


## Optuna でハイパーパラメータ探索

時系列 4-fold の **平均 AUC** を最大化する。`n_trials` を増やすと探索時間は伸びるが、当たりが出やすくなる。

In [3]:
EARLY_STOPPING_ROUNDS = 30
N_TRIALS = 50  # 試行数（時間があれば 80〜100 に増やす）

def run_cv_with_params(params):
    """時系列 4-fold CV を実行し、fold 平均 AUC を返す。"""
    fold_scores = []
    for tr_idx, val_idx in time_splits:
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)]
        )
        pred = model.predict_proba(X_val)[:, 1]
        fold_scores.append(roc_auc_score(y_val, pred))
    return np.mean(fold_scores)

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "random_state": 42,
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
    }
    return run_cv_with_params(params)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=15))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest CV AUC: {study.best_value:.4f}")
print("Best params:", study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-03-02 23:24:33,833] Trial 11 failed with parameters: {'n_estimators': 1088, 'learning_rate': 0.05932597997400133, 'num_leaves': 29, 'min_data_in_leaf': 146, 'feature_fraction': 0.8803925243084487, 'reg_alpha': 0.17583640270008521, 'reg_lambda': 1.2130221181165162, 'max_depth': 8} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/komatsuzakiharutoshi/Desktop/ds_dojo4/.venv/lib/python3.9/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/j4/bdb3d1w925qgn2ljkfxwcwjw0000gn/T/ipykernel_63569/2036042737.py", line 36, in objective
    return run_cv_with_params(params)
  File "/var/folders/j4/bdb3d1w925qgn2ljkfxwcwjw0000gn/T/ipykernel_63569/2036042737.py", line 11, in run_cv_with_params
    model.fit(
  File "/Users/komatsuzakiharutoshi/Desktop/ds_dojo4/.venv/lib/python3.9/site-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/Users/komatsu

KeyboardInterrupt: 

In [ ]:
# ベストパラメータを LGB 用に整形
lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "random_state": 42, "verbosity": -1,
    **study.best_params,
}
# early_stopping が効くので n_estimators は多めでよい（Optuna は 200〜1200 を探索）
lgb_params["n_estimators"] = max(lgb_params.get("n_estimators", 500), 400)

# チューニング結果で改めて CV（fold ごとのスコア表示）
oof = np.zeros(len(train))
test_preds = []
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    fold_scores.append(fold_auc)
    test_preds.append(model.predict_proba(X_test)[:, 1])
    print(f"Fold{fold+1}: val_year={VAL_YEARS[fold]}, AUC={fold_auc:.4f}")

print(f"\nCV AUC (fold mean): {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

In [ ]:
# 提出用: 全 train で学習 → 予測
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X, y)
final_pred = model_full.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"ID": test["ID"], "target": final_pred})
out_path = os.path.join(OUTPUT_DIR, "submissions", "submission_lgb_tuned.csv")
submission.to_csv(out_path, index=False)
print(f"Saved {out_path} (rows: {len(submission):,})")

# 特徴量重要度
importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_full.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance_df)
print(f"\n重要度 Top5: {list(importance_df.head(5)['feature'].values)}")